# Data Review

本脚本批量处理多个被试的 CSV 文件（文件名pattern：`创意用途生成任务_*.csv`），根据以下规则标记可疑被试（以实际代码为准）：

- **无效回答**：乱码、无意义字符居多，清洗后字符数量少于3
- **反应时过短**：<2秒且低于全局5%分位数
- **语义相似重复**：同一物品内答案高度相似（>0.5）
- **答案数量不足**：某物品答案少于3个
- **末尾无动作**：任务结束前120s无任何操作
- **AI依赖**：答案与AI提示高度相似（>0.5）的占比达到50%

输出：一个包含所有被试ID及各规则命中情况的 CSV 文件。

## 1. 导入库

In [ ]:
# 基本库
import pandas as pd
import numpy as np
import re
import glob
import os
import json

# 文本相似度（使用 TF-IDF + 余弦相似度）
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. 常量

In [ ]:
# 反应时阈值（秒）
RT_THRESHOLD = 2.0
# 全局反应时百分位数（用于标记“过快”）
RT_PERCENTILE = 5  # 5% 分位数

# 每个答案最少字符长度
MIN_CHAR_PER_ANSWER = 2

# 每个物品最少答案数量
MIN_ANSWERS_PER_ITEM = 3

# 末尾无动作阈值（秒）
INACTIVE_THRESHOLD = 120

# 语义相似度阈值
SEMANTIC_SIM_THRESHOLD = 0.5

# AI依赖比例阈值
AI_DEPEND_THRESHOLD = 0.5

# 忽略的练习物品名称
PRACTICE_ITEMS = ['肥皂']

# 输入文件路径的正则匹配pattern
FILE_PATTERN = "../data/to_review/创意用途生成任务_*.csv"

# 输出文件名
OUTPUT_CSV = "data_review.csv"

## 3. 工具函数

In [ ]:
def is_garbage(text):
    """
    判断文本是否为无意义内容（乱码、表情、纯标点等）
    规则：
    - 去除常见标点符号和空格后，剩余字符长度 < MIN_ANSWERS_PER_ITEM
    - 且不包含任何中文字符
    - 或者文本在黑名单中（如“不知道”、“无”等）
    """
    if not isinstance(text, str):
        return True
    text = text.strip()
    if text == "":
        return True

    # 黑名单
    blacklist = {"不知道", "无", "没有", "null", "none", "?", "？？", "。。", "...", "……"}
    if text in blacklist:
        print('黑名单：', text)
        return True

    # 去除常见标点符号（中英文）
    cleaned = re.sub(r'[^\w\u4e00-\u9fff]', '', text)  # 保留字母、数字、中文
    if len(cleaned) < MIN_ANSWERS_PER_ITEM and not re.search(r'[\u4e00-\u9fff]', cleaned):
        print('标点：', text)
        return True

    return False

def extract_messages(msg_str):
    """
    解析 messages 字段（JSON字符串），返回 (user_messages, assistant_messages)
    每条消息格式：{"role": "user"/"assistant", "content": "文本", "time": 秒数}
    """
    user_msgs = []
    assistant_msgs = []
    if pd.isna(msg_str) or msg_str == "" or msg_str == "[]":
        return user_msgs, assistant_msgs
    try:
        msgs = json.loads(msg_str)
        for m in msgs:
            if m.get("role") == "user":
                user_msgs.append({"content": m.get("content", ""), "time": m.get("time", 0)})
            elif m.get("role") == "assistant":
                assistant_msgs.append({"content": m.get("content", ""), "time": m.get("time", 0)})
    except:
        pass
    return user_msgs, assistant_msgs

def parse_ai_clicks(click_str):
    """
    解析 aiClickEvents 字段，返回点击时间列表（秒）
    字段格式示例: [{"type": "click", "time": 111}, ...]
    """
    if pd.isna(click_str) or click_str == "" or click_str == "[]":
        return []
    try:
        events = json.loads(click_str)
        if isinstance(events, list):
            # 提取每个事件的 time 字段，过滤掉没有 time 的
            times = [event['time'] for event in events if 'time' in event]
            return times
        else:
            return []
    except:
        return []
    
def parse_key_rt(rt_str, aut_start):
    """
    解析 KeyRespRecord.rt 字段，返回绝对时间列表（秒）
    字段示例: "[7.6576, 8.9740, ...]"
    aut_start: 当前物品的开始时间（绝对时间），用于将相对时间转换为绝对时间
    """
    if pd.isna(rt_str) or rt_str == "" or rt_str == "[]":
        return []
    try:
        times = json.loads(rt_str)
        if isinstance(times, list):
            # 过滤掉 None 并转换为浮点数，然后加上 aut_start 得到绝对时间
            abs_times = [aut_start + float(t) for t in times if t is not None]
            return abs_times
        else:
            return []
    except:
        return []

def compute_similarity(texts, threshold=SEMANTIC_SIM_THRESHOLD):
    """
    计算文本列表两两之间的余弦相似度，返回最大相似度（排除自身）
    如果最大相似度超过 threshold，打印出相似句子对（DEBUG）
    """
    if len(texts) < MIN_ANSWERS_PER_ITEM:
        return 0
    # 过滤空文本
    texts = [t if isinstance(t, str) and t.strip() != "" else "空" for t in texts]
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1,3))
    try:
        tfidf = vectorizer.fit_transform(texts)
        sim_matrix = cosine_similarity(tfidf)
        np.fill_diagonal(sim_matrix, 0)
        max_sim = sim_matrix.max()
        
        if max_sim > threshold:
            # 找到所有达到最大相似度的位置（可能有多对）
            indices = np.where(sim_matrix == max_sim)
            # 取第一对（i < j 确保不重复打印）
            for i, j in zip(indices[0], indices[1]):
                if i < j:  # 避免打印(i,j)和(j,i)重复
                    print(f"* [语义相似] 相似度：{max_sim:.3f} (>{threshold})，句子对：")
                    print(f"  句子1: {texts[i]}")
                    print(f"  句子2: {texts[j]}")
                    break  # 只打印第一对即可
        return max_sim
    except Exception as e:
        print(f"[相似度计算异常] {e}")
        return 0

def compute_pair_similarity(text1, text2):
    """计算两个文本的相似度"""
    if not isinstance(text1, str) or not isinstance(text2, str):
        return 0
    if text1.strip() == "" or text2.strip() == "":
        return 0
    try:
        vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1,3))
        tfidf = vectorizer.fit_transform([text1, text2])
        sim = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
        if sim > SEMANTIC_SIM_THRESHOLD:
            print(f"^ [AI相似] 相似度：{sim:.3f} (>{SEMANTIC_SIM_THRESHOLD})，文本：\"{text1}\" <-> \"{text2}\"")
        return sim
    except:
        return 0

## 4. 单被试处理方法

In [ ]:
def process_one_file(filepath):
    """
    处理单个CSV文件，返回一个字典：被试ID及各规则是否命中（True/False）
    """
    df = pd.read_csv(filepath, encoding='utf-8-sig')
    
    # 提取被试ID（文件名中的hash部分）
    subject_id = os.path.basename(filepath).replace('创意用途生成任务_', '').replace('.csv', '')
    
    # 初始化规则标记
    flags = {
        # 无效回答
        'invalid_response': False,
        # 回答过快
        'too_fast': False,
        # 语义重复
        'semantic_duplicate': False,
        # 答案数量过少
        'insufficient_answers': False,
        # 反应过长
        'long_inactive': False,
        # 与AI高度重复
        'ai_dependent': False,
    }

    # 初始化AI依赖统计
    total_answers = 0
    ai_dependent_answers = 0
    
    # 收集所有答案的反应时（用于全局百分位数）
    all_rts = []
    
    # 按物品循环（每个试次对应一行）
    # 注意：有些行可能是练习或其他阶段，我们根据 itemName 非空且不在忽略列表中筛选
    item_rows = df[df['itemName'].notna() & ~df['itemName'].isin(PRACTICE_ITEMS)]
    
    for idx, row in item_rows.iterrows():
        item = row['itemName']
        aut_start = row['AUT.started']
        aut_stop = row['AUT.stopped']
        
        # 解析 messages
        user_msgs, assistant_msgs = extract_messages(row['messages'])
        
        # 没有任何user回答，标记为数量不足，并跳过
        if not user_msgs:
            print(f"* [答案数量不足] 物品：{item}，没有任何回答")
            flags['insufficient_answers'] = True
            continue
        
        # 对答案按时间排序
        user_msgs_sorted = sorted(user_msgs, key=lambda x: x['time'])
        answers = [msg['content'] for msg in user_msgs_sorted]
        answer_times = [msg['time'] for msg in user_msgs_sorted]
        
        # 计算每个答案的反应时
        rts = []
        prev_time = aut_start   # 上一个事件时间（开始或上一次提交）
        for t in answer_times:
            rt = t - prev_time  # 当前答案的反应时
            rts.append(rt)
            prev_time = t       # 更新为本次提交时间，供下一个答案使用
        all_rts.extend(rts)
        
        # ---------- 规则1：无效回答 ----------
        for ans in answers:
            if is_garbage(ans):
                flags['invalid_response'] = True
                break
        
        # ---------- 规则2：反应时过短（暂不标记，等全局分位数）----------
        # 先收集rts，后面统一处理
        
        # ---------- 规则3：语义相似重复 ----------
        if compute_similarity(answers) > SEMANTIC_SIM_THRESHOLD:
            flags['semantic_duplicate'] = True
        
        # ---------- 规则4：答案数量不足 ----------
        if len(answers) < MIN_ANSWERS_PER_ITEM:
            print(f"* [答案数量不足] 物品：{item}，数量：{len(answers)} (<{MIN_ANSWERS_PER_ITEM})")
            flags['insufficient_answers'] = True
        
        # ---------- 规则5：末尾无动作 ----------
        aut_start = row['AUT.started']
        events = []  # 存储 (时间, 事件类型) 元组

        # 答案提交
        for t in answer_times:
            events.append((t, "answer"))
        # AI点击
        ai_clicks = parse_ai_clicks(row['aiClickEvents'])
        for t in ai_clicks:
            events.append((t, "ai_click"))
        # AI回复
        for msg in assistant_msgs:
            events.append((msg['time'], "ai_reply"))
        # 键盘事件（转换为绝对时间）
        key_abs_times = parse_key_rt(row['KeyRespRecord.rt'], aut_start)
        for t in key_abs_times:
            events.append((t, "keyboard"))

        if events:
            # 找到时间最大的事件
            last_event = max(events, key=lambda x: x[0])
            last_event_time, last_event_type = last_event
            if aut_stop - last_event_time > INACTIVE_THRESHOLD:
                print(f"* [末尾无动作] 物品：{item}，最后事件时间：{last_event_time:.2f}s ({last_event_type})，结束时间：{aut_stop:.2f}s，间隔：{aut_stop-last_event_time:.2f}s (>{INACTIVE_THRESHOLD}s)")
                flags['long_inactive'] = True
        
        # ---------- 规则6：AI依赖（统计占比） ----------
        if ai_clicks and assistant_msgs:
            # 排序
            ai_clicks_sorted = sorted(ai_clicks)
            assistant_msgs_sorted = sorted(assistant_msgs, key=lambda x: x['time'])
            user_msgs_sorted = sorted(user_msgs, key=lambda x: x['time'])

            # 获取答案数量（用于统计）
            total_answers += len(user_msgs_sorted)
            
            # 记录该物品中哪些答案被判定为依赖AI
            matched_indices = set()
            
            # 对每个AI点击，尝试找到对应的assistant回复和随后的用户答案
            for click_t in ai_clicks_sorted:
                # 找最近的assistant消息（在click_t之后，且时间差<5秒）
                suitable_assistant = None
                for a in assistant_msgs_sorted:
                    if a['time'] > click_t and a['time'] - click_t < 5:
                        suitable_assistant = a
                        break
                if suitable_assistant is None:
                    continue
                # 找该assistant之后最近的user消息（时间差<10秒）
                suitable_user = None
                user_index = None
                for i, u in enumerate(user_msgs_sorted):
                    if u['time'] > suitable_assistant['time']: # and u['time'] - suitable_assistant['time'] < 10:
                        suitable_user = u
                        user_index = i
                        break
                if suitable_user is None:
                    continue
                # 计算相似度
                sim = compute_pair_similarity(suitable_user['content'], suitable_assistant['content'])
                if sim > SEMANTIC_SIM_THRESHOLD:
                    matched_indices.add(user_index)   # 记录答案索引，避免重复
                    # 注意：这里不break，继续检查其他点击，以发现更多依赖答案
            
            # 更新全局计数
            ai_dependent_answers += len(matched_indices)

    # 判断AI依赖比例是否超过50%
    if total_answers > 0:
        print(f"^ [AI 依赖比例] {ai_dependent_answers / total_answers}")
        if (ai_dependent_answers / total_answers) > AI_DEPEND_THRESHOLD:
            print(f"* [高度依赖 AI] 依赖比例：{ai_dependent_answers / total_answers} (>{AI_DEPEND_THRESHOLD})")
            flags['ai_dependent'] = True
    
    # ---------- 全局反应时过短标记 ----------
    if all_rts:
        # 计算所有答案反应时的5%分位数
        rt_percentile = np.percentile(all_rts, RT_PERCENTILE)
        # 对每个物品的每个答案，如果反应时<2秒且<该分位数，则标记
        # 重新遍历一次（简便起见，我们可以在之前收集时记录，但为了代码清晰，重新从原始行计算）
        # 由于已经遍历过一次，我们再次读取数据（或者优化：在第一次遍历时记录每个rts对应的标记，但简单起见重新处理）
        # 这里为了效率，我们重新读取一次（数据量不大，可以接受）
        for idx, row in item_rows.iterrows():
            aut_start = row['AUT.started']
            user_msgs, _ = extract_messages(row['messages'])
            if not user_msgs:
                continue
            # 排序
            user_msgs_sorted = sorted(user_msgs, key=lambda x: x['time'])
            answer_times = [msg['time'] for msg in user_msgs_sorted]
            
            prev_time = aut_start
            for t in answer_times:
                rt = t - prev_time
                if rt < RT_THRESHOLD and rt < rt_percentile:
                    flags['too_fast'] = True
                    break
                prev_time = t
            if flags['too_fast']:
                break
    
    # 如果任何规则命中，则整体可疑
    flags['suspect'] = any(flags.values())
    
    # 返回结果
    result = {'subject_id': subject_id}
    result.update(flags)
    return result

## 5. 批量处理

In [ ]:
def main():
    files = glob.glob(FILE_PATTERN)
    if not files:
        print(f"未找到匹配 {FILE_PATTERN} 的文件")
        return
    
    results = []
    for f in files:
        print(f"\n处理: {f}")
        try:
            res = process_one_file(f)
            results.append(res)
        except Exception as e:
            print(f"处理文件 {f} 时出错: {e}")
            # 仍添加一条记录
            results.append({
                'subject_id': os.path.basename(f).replace('创意用途生成任务_', '').replace('.csv', ''),
                'invalid_response': False,
                'too_fast': False,
                'semantic_duplicate': False,
                'insufficient_answers': False,
                'long_inactive': False,
                'ai_dependent': False,
                'suspect': True,  # 出错视为可疑
                'error': str(e)
            })
    
    # 创建DataFrame并保存
    df_out = pd.DataFrame(results)
    # 按需调整列顺序
    cols = ['subject_id', 'invalid_response', 'too_fast', 'semantic_duplicate',
            'insufficient_answers', 'long_inactive', 'ai_dependent', 'suspect']
    if 'error' in df_out.columns:
        cols.append('error')
    df_out = df_out[cols]
    df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"完成！结果已保存至 {OUTPUT_CSV}")
    
    # 显示简要统计
    print("\n=== 统计 ===")
    print(f"总被试数: {len(df_out)}")
    print(f"可疑被试数: {df_out['suspect'].sum()}")
    for col in ['invalid_response', 'too_fast', 'semantic_duplicate', 'insufficient_answers', 'long_inactive', 'ai_dependent']:
        print(f"{col}: {df_out[col].sum()}")

if __name__ == "__main__":
    main()